In [ ]:
file_path=""

"""Parameters"""

In [ ]:
print(f"Received file_path: '{file_path}'")
assert file_path != "", "file_path is empty - parameter not passed correctly"

In [ ]:
date_columns = [
    "expected_graduation", "enroll_date", "birth_date", "session_date", "enrolled_at", "start_date", "end_date", "end_date", "start_date", "created_at", "updated_at", "submitted_at", "hire_date", "scheduled_date"
]
def transform(df):
    existing_date_cols = [c for c in date_columns if c in df.columns]
    for c in existing_date_cols:
        df = df.withColumn(c, to_timestamp(col(c)))

    return df

def convert_datetime_to_date_keys(df):
    datetime_cols = [col_name for col_name, data_type in df.dtypes if data_type in ("timestamp", "date")]

    for col_name in datetime_cols:
        df = df.withColumn(
           f"{col_name}_key",
            date_format(col(col_name), "yyyyMMdd").cast("int")
        )
    df = df.drop(*datetime_cols)
    return df

In [ ]:
df = transform(df)
df = convert_datetime_to_date_keys(df)

In [ ]:
from datetime import date


storage_container_name = ""
storage_account_name = ""

table = file_path.split("/bronze/")[1].split("/")[0].replace(".parquet", "")

today_str    = date.today().strftime("%Y-%m-%d")
tmp_path     = f"abfss://{storage_account_name}@{storage_container_name}.dfs.core.windows.net/silver/{table}_tmp/"
dated_folder = f"abfss://{storage_account_name}@{storage_container_name}.dfs.core.windows.net/silver/{table}/{today_str}/"
final_file   = f"{dated_folder}{table}.parquet"
safe_path    = file_path.replace("'", "\\'")

try:
    df.coalesce(1).write.mode("overwrite").parquet(tmp_path)
    files     = notebookutils.fs.ls(tmp_path)
    part_file = [f.path for f in files if f.name.endswith(".parquet")][0]
    notebookutils.fs.mkdirs(dated_folder)
    notebookutils.fs.mv(part_file, final_file)
    notebookutils.fs.rm(tmp_path, recurse=True)

    spark.sql(f"""
        UPDATE control.pending_files
        SET status = 'done'
        WHERE file_path = '{safe_path}'
    """)

    spark.sql(f"""
        UPDATE control.latest_tables
        SET is_latest = 'false'
        WHERE is_latest = 'true' AND table_name = '{table}'
    """)

    spark.sql(f"""
        INSERT INTO control.latest_tables
        VALUES ('{table}', '{final_file}', current_timestamp(), 'true')
    """)

    print(f"Written to: {final_file}")
    print(f"Marked as done: {file_path}")

except Exception as e:
    spark.sql(f"""
        UPDATE control.pending_files
        SET status = 'pending'
        WHERE file_path = '{safe_path}'
    """)
    print(f"Failed to process: {file_path}")
    raise e
